In [4]:
from api import InstructorLLM
from pydantic import BaseModel
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

In [5]:
STORE = Path("../store/inhouse")
STORE.mkdir(parents=True, exist_ok=True)
RAW_CSV = STORE / "raw.csv"
OUT_CSV = STORE / "raw.csv"

# test controls
MAX_CONCEPTS = None      # set back to None for full run
MAX_SUBTOPICS = None     # set back to None for full run

In [6]:
TOPICS = {
    "bacteria": [
        "Cell structure and morphology",
        "Gram-positive vs gram-negative bacteria",
        "Binary fission and reproduction",
        "Antibiotic resistance mechanisms",
        "Bacterial flagella and motility",
        "Biofilm formation",
        "Pathogenic bacteria species",
        "Beneficial gut bacteria",
        "Bacterial taxonomy and classification",
        "Endospore formation",
        "Bacterial genetics and plasmids",
        "Conjugation and horizontal gene transfer",
        "Bacterial toxins",
        "Nitrogen-fixing bacteria",
        "Cyanobacteria and photosynthesis",
        "Extremophile bacteria",
        "Bacterial cell wall composition",
        "Quorum sensing in bacteria",
        "Bacterial diseases in humans",
        "Bacterial metabolism types",
        "Aerobic vs anaerobic bacteria",
        "Bacterial size and scale",
        "History of bacteriology",
        "Famous bacteriologists",
        "Bacteria in food production",
        "Bacterial decomposition",
        "Bacteria in water treatment",
        "Bacterial bioluminescence",
        "Mycobacteria facts",
        "Streptococcus species",
        "Staphylococcus species",
        "E. coli facts",
        "Salmonella facts",
        "Tuberculosis bacterium",
        "Bacterial vaccines",
        "Probiotic bacteria",
        "Bacteria vs viruses differences",
        "Bacterial colonies and culture techniques",
        "Bacterial staining techniques",
        "Bacterial ribosomes",
        "Bacterial DNA replication",
        "Chemotaxis in bacteria",
        "Bacterial symbiosis",
        "Bacteria in soil ecology",
        "Bacterial infection treatment history",
        "Bacterial genetic engineering uses",
        "CRISPR origins in bacteria",
        "Bacterial membrane transport",
        "Archaea vs bacteria",
        "Bacterial role in carbon cycle",
    ],
    "cats": [
        "Domestic cat breeds",
        "Cat anatomy and skeleton",
        "Cat senses and perception",
        "Cat behavior and body language",
        "Cat purring mechanics",
        "Cat diet and nutrition",
        "Cat reproduction and kittens",
        "Cat grooming habits",
        "Cat sleep patterns",
        "Cat hunting instincts",
        "Cat domestication history",
        "Cat lifespan and aging",
        "Cat coat colors and genetics",
        "Cat eye colors and vision",
        "Cat whiskers function",
        "Cat claws and scratching behavior",
        "Cat teeth and dental health",
        "Cat tail communication",
        "Cat vocalizations and meowing",
        "Famous cats in history",
        "Cats in ancient Egypt",
        "Cat health and common diseases",
        "Cat vaccinations",
        "Cat parasites and fleas",
        "Cat allergies in humans",
        "Wild cat species",
        "Big cats vs domestic cats",
        "Cat intelligence and learning",
        "Cat territorial behavior",
        "Cats and water relationship",
        "Cat breeds origin countries",
        "Cat show competitions",
        "Cat population statistics worldwide",
        "Cats in literature and art",
        "Cat superstitions and folklore",
        "Cat genetics and inheritance",
        "Indoor vs outdoor cats",
        "Cat social structure",
        "Cat play behavior",
        "Cat digestive system",
        "Cat respiratory system",
        "Cat cardiovascular system",
        "Cat nervous system",
        "Cats and toxoplasmosis",
        "Cat rescue and adoption",
        "Cat training techniques",
        "Oldest cat breeds",
        "Cat size and weight ranges",
        "Cat memory and cognition",
        "Cats in internet culture",
    ],
    "obama": [
        "Early life and birthplace",
        "Family background and parents",
        "Education at Punahou School",
        "Columbia University years",
        "Harvard Law School years",
        "Community organizing in Chicago",
        "Teaching at University of Chicago",
        "Illinois state senate career",
        "2004 DNC keynote speech",
        "2008 presidential campaign",
        "2008 election results",
        "First term cabinet members",
        "Affordable Care Act",
        "Economic stimulus package 2009",
        "Osama bin Laden raid",
        "Obama foreign policy and diplomacy",
        "Obama Nobel Peace Prize",
        "2012 reelection campaign",
        "Second term policy achievements",
        "Michelle Obama partnership",
        "Obama daughters Sasha and Malia",
        "Obama White House pets",
        "Obama executive orders",
        "Obama Supreme Court nominations",
        "Obama climate change policies",
        "Obama immigration policies",
        "Cuba relations normalization under Obama",
        "Iran nuclear deal under Obama",
        "Obama speeches and rhetoric style",
        "Books authored by Obama",
        "Obama post-presidency activities",
        "Obama Foundation",
        "Obama Presidential Library",
        "Obama relationship with Congress",
        "Obama administration controversies",
        "Obama technology and innovation policies",
        "Obama education policies",
        "Obama healthcare reform details",
        "Obama military decisions",
        "Obama and civil rights",
        "Obama legislative record",
        "Obama campaign fundraising strategy",
        "Obama approval ratings over time",
        "Obama pardons and commutations",
        "Obama State of the Union addresses",
        "Obama relations with world leaders",
        "Obama hobbies and personal interests",
        "Awards and honors received by Obama",
        "Obama cultural impact",
        "Obama political legacy",
    ],
    "people": [
        "Human body organ systems",
        "Human brain and cognition",
        "Human DNA and genetics",
        "Human evolution timeline",
        "Early human species",
        "Human population milestones",
        "Human lifespan across history",
        "Human senses overview",
        "Human reproduction biology",
        "Human blood types",
        "Human skeletal system",
        "Human muscular system",
        "Human immune system",
        "Human sleep science",
        "Human nutrition requirements",
        "Human language origins",
        "Human memory and learning",
        "Human emotions and psychology",
        "Human skin and hair",
        "Human teeth and dental facts",
        "Human eye anatomy",
        "Human hearing and ear anatomy",
        "Human cardiovascular system",
        "Human respiratory system",
        "Human digestive system",
        "Human nervous system",
        "Human endocrine system",
        "Human growth and development stages",
        "Human diseases and historic epidemics",
        "Human migration patterns",
        "Human cultural universals",
        "Human tool use history",
        "Human agriculture origins",
        "Human writing systems history",
        "Human record-breaking physical feats",
        "Human demographic statistics",
        "Human pregnancy and birth",
        "Human child development stages",
        "Human aging biology",
        "Human circadian rhythms",
        "Human body temperature regulation",
        "Human reflexes and instincts",
        "Human microbiome",
        "Human genetic disorders",
        "Human pain perception",
        "Human creativity and art origins",
        "Human cooperation and social behavior",
        "Human speech and vocal anatomy",
        "Human hand and grip evolution",
        "Human water and hydration needs",
    ],
    "dogs": [
        "Dog breed classifications",
        "Dog anatomy and skeleton",
        "Dog senses and perception",
        "Dog behavior and body language",
        "Dog barking and vocalizations",
        "Dog diet and nutrition",
        "Dog reproduction and puppies",
        "Dog grooming and coat types",
        "Dog sleep patterns",
        "Dog training methods",
        "Dog domestication history",
        "Dog lifespan by breed",
        "Dog coat colors and genetics",
        "Dog intelligence rankings",
        "Dog health and common diseases",
        "Dog vaccinations",
        "Dog parasites and ticks",
        "Famous dogs in history",
        "Dog breeds origin countries",
        "Working dogs and service dogs",
        "Police and military dogs",
        "Sled dogs and racing",
        "Dog shows and competitions",
        "Dog size extremes by breed",
        "Dog teeth and dental care",
        "Dog tail docking and ear cropping history",
        "Dog nose and scent ability",
        "Dog paw anatomy",
        "Dog swimming ability by breed",
        "Dog social hierarchy and pack behavior",
        "Dog aging and senior care",
        "Dog rescue and adoption",
        "Dog laws and regulations",
        "Dog population statistics worldwide",
        "Dogs in space exploration",
        "Therapy and emotional support dogs",
        "Guide dogs for the blind",
        "Dog breeding ethics",
        "Dog food industry",
        "Dog allergies and sensitivities",
        "Dog exercise requirements by breed",
        "Dog communication with humans",
        "Dog memory and cognition",
        "Dog genetic diversity",
        "Ancient dog breeds",
        "Dog skeletal differences by breed",
        "Dog cardiovascular system",
        "Dog digestive system",
        "Herding dog breeds and behavior",
        "Dogs in literature and film",
    ],
    "paris": [
        "Paris founding and early history",
        "Paris geography and the Seine river",
        "Eiffel Tower facts",
        "Notre-Dame Cathedral facts",
        "The Louvre Museum facts",
        "Paris arrondissements system",
        "Paris Metro system",
        "Paris population and demographics",
        "Champs-Élysées boulevard facts",
        "Arc de Triomphe facts",
        "Sacré-Cœur Basilica facts",
        "Moulin Rouge history",
        "Paris fashion industry",
        "Paris cuisine and famous restaurants",
        "Paris cafés and café culture",
        "Paris in World War II",
        "French Revolution events in Paris",
        "Paris Commune of 1871",
        "Haussmann renovation of Paris",
        "Paris parks and gardens",
        "Luxembourg Gardens facts",
        "Tuileries Garden facts",
        "Père Lachaise Cemetery facts",
        "Paris bridges over the Seine",
        "Paris catacombs facts",
        "Musée d'Orsay facts",
        "Centre Pompidou facts",
        "Paris universities and the Sorbonne",
        "Paris literary history",
        "Paris art movements",
        "Paris music scene history",
        "Paris theater and opera history",
        "Paris Olympics history",
        "Paris climate and weather",
        "Paris transportation systems",
        "Paris architecture styles",
        "Paris nightlife history",
        "Palace of Versailles connection to Paris",
        "Paris airports",
        "Paris street names and their history",
        "Paris festivals and annual events",
        "Paris economy and industries",
        "Paris governance and mayors",
        "Seine river facts and history",
        "Paris and the Enlightenment",
        "Paris bookshops and publishing history",
        "Paris film and cinema history",
        "Paris sports teams",
        "Paris immigrant communities",
        "Paris tourism statistics",
    ],
    "lasers": [
        "Laser invention and history",
        "How lasers work and stimulated emission",
        "Types of lasers (gas, solid-state, semiconductor)",
        "Laser wavelengths and spectrum",
        "Laser safety classifications",
        "Lasers in medicine and surgery",
        "LASIK eye surgery with lasers",
        "Lasers in manufacturing and cutting",
        "Lasers in telecommunications",
        "Laser pointer facts",
        "Lasers in barcode scanners",
        "Laser printer technology",
        "Lasers in entertainment and light shows",
        "Laser holography",
        "Laser interferometry",
        "Lasers in military applications",
        "Laser-guided weapons",
        "Lasers in space exploration",
        "Laser cooling of atoms",
        "Laser spectroscopy",
        "Ruby laser history",
        "CO2 laser facts",
        "Helium-neon laser facts",
        "Semiconductor diode lasers",
        "Fiber laser technology",
        "Ultrafast femtosecond lasers",
        "Laser power measurements and units",
        "Laser beam properties",
        "Laser coherence properties",
        "Laser cavities and mirrors",
        "Nobel Prizes related to lasers",
        "Laser engraving and etching",
        "Lasers in 3D printing",
        "LIDAR and laser range finding",
        "Lasers in CD DVD and Blu-ray players",
        "Laser hair removal technology",
        "Laser tattoo removal",
        "Laser skin treatments",
        "Lasers in dentistry",
        "Laser welding technology",
        "Laser alignment tools",
        "Laser communication systems",
        "Quantum cascade lasers",
        "Excimer laser facts",
        "Laser acronym origin and meaning",
        "Maser vs laser differences",
        "Lasers in nuclear fusion research",
        "Famous laser researchers and pioneers",
        "Laser speed and light properties",
        "Laser applications in agriculture",
    ],
    "united_states": [
        "US founding and Declaration of Independence",
        "US Constitution facts",
        "US Bill of Rights",
        "US states count and admission order",
        "US state capitals",
        "US presidents list and facts",
        "US flag history and symbolism",
        "US national anthem facts",
        "US population statistics",
        "US major geographic features",
        "US Civil War key facts",
        "US Revolutionary War facts",
        "US territorial expansion history",
        "US Supreme Court landmark cases",
        "US Electoral College system",
        "US Congress structure and facts",
        "US federal holidays",
        "US national parks system",
        "US economy and GDP facts",
        "US military branches",
        "US space program and NASA",
        "US immigration history",
        "US civil rights movement",
        "US Constitutional Amendments",
        "US currency and Federal Reserve",
        "US education system",
        "US healthcare system facts",
        "US transportation infrastructure",
        "US invention and innovation history",
        "US sports and major leagues",
        "US music genres that originated in America",
        "US film industry and Hollywood",
        "US agriculture and farming facts",
        "US natural disasters history",
        "US climate and weather patterns",
        "US major rivers and waterways",
        "US mountain ranges",
        "US largest cities by population",
        "US time zones",
        "US Native American history",
        "US Prohibition era facts",
        "US Great Depression facts",
        "US Cold War involvement",
        "US involvement in World Wars",
        "US tech industry and Silicon Valley",
        "US energy production facts",
        "US international relations",
        "US social movements history",
        "US census and demographic trends",
        "US cultural landmarks and monuments",
    ],
    "the_moon": [
        "Moon formation theories",
        "Moon size and distance from Earth",
        "Moon surface features and craters",
        "Moon phases and lunar cycle",
        "Moon effect on ocean tides",
        "Moon gravity compared to Earth",
        "Moon lack of atmosphere",
        "Moon temperature extremes",
        "Apollo 11 Moon mission",
        "Apollo program Moon missions",
        "First humans on the Moon",
        "Moon landing conspiracy theories debunked",
        "Lunar soil regolith composition",
        "Moon rocks brought to Earth",
        "Moon far side facts",
        "Lunar eclipse facts",
        "Moon orbit and rotation",
        "Moon age estimates",
        "Moon maria (seas) features",
        "Moon mountain ranges",
        "Lunar exploration before Apollo",
        "Soviet Luna program facts",
        "Chinese Chang'e lunar missions",
        "Artemis Moon program",
        "Moon magnetic field",
        "Moon internal structure",
        "Water ice on the Moon",
        "Moon influence on calendars",
        "Moon in mythology and culture",
        "Full moon names by month",
        "Supermoons and blood moons",
        "Moon effect on animal behavior",
        "Moonlight properties",
        "Lunar rovers and vehicles",
        "Moonquakes and lunar seismic activity",
        "Future Moon colonization plans",
        "Moon resource mining potential",
        "Astronauts who walked on the Moon",
        "Moon photography history",
        "Moon libration phenomenon",
        "Moon recession from Earth",
        "Lunar Gateway space station",
        "Moon albedo and brightness",
        "Transient lunar phenomena",
        "Moon in literature and poetry",
        "Moon synchronous rotation",
        "Lunar timekeeping systems",
        "Moon treaties and space law",
        "Moon impact on Earth rotation",
        "Notable lunar scientists and astronomers",
    ],
    "chess": [
        "Chess origins and history",
        "Chess piece movements and rules",
        "Chess board setup and notation",
        "Famous chess world champions",
        "Chess openings and theory",
        "Chess endgame principles",
        "Chess middlegame strategy",
        "Chess tactics: forks pins and skewers",
        "Chess Elo rating system",
        "Chess tournaments and major competitions",
        "Chess clocks and time controls",
        "Bobby Fischer chess facts",
        "Garry Kasparov chess facts",
        "Magnus Carlsen chess facts",
        "Deep Blue vs Kasparov match",
        "Chess engines and AI",
        "Chess variants: blitz rapid and bullet",
        "Chess notation systems",
        "Castling rules in chess",
        "En passant rule in chess",
        "Stalemate and draw rules in chess",
        "Chess grandmaster title requirements",
        "FIDE chess organization",
        "Chess Olympiad facts",
        "Chess in education",
        "Chess and mathematics connections",
        "Famous chess games in history",
        "Chess prodigies",
        "Women in chess history",
        "Chess set design history",
        "Chess terminology and vocabulary",
        "Pawn promotion rules in chess",
        "Chess strategy principles",
        "Chess pattern recognition",
        "Classic chess books",
        "Chess in film and television",
        "Chess in literature",
        "Online chess platforms",
        "Chess puzzle types",
        "Chess etiquette and sportsmanship",
        "Chess and cognitive science research",
        "Simultaneous chess exhibition matches",
        "Blindfold chess",
        "Correspondence chess history",
        "Chess composition and problems",
        "Chess national federations",
        "Speed chess records",
        "Chess scandals and controversies",
        "Chess in different cultures",
        "Ancient predecessors of chess",
    ],
}

## Generation

In [ ]:
llm = InstructorLLM(provider_model="openai/gpt-5-nano", concurrency=500)

In [ ]:
class QAItem(BaseModel):
    question: str
    answer: str


class QASet(BaseModel):
    q1: QAItem
    q2: QAItem
    q3: QAItem
    q4: QAItem
    q5: QAItem
    q6: QAItem
    q7: QAItem
    q8: QAItem
    q9: QAItem
    q10: QAItem
    q11: QAItem
    q12: QAItem
    q13: QAItem
    q14: QAItem
    q15: QAItem
    q16: QAItem
    q17: QAItem
    q18: QAItem
    q19: QAItem
    q20: QAItem

In [ ]:
CONCEPT_GUIDANCE = {
    "bacteria": "Use bacteria-specific words such as bacterial, bacterium, microbe, gram-positive, gram-negative, plasmid, or biofilm. Do not ask generic biology questions that could fit animals, humans, or viruses.",
    "cats": "Use cat, feline, kitten, whiskers, purring, or named cat breeds. Do not ask generic pet or mammal questions that could apply to dogs or people.",
    "obama": "Use Barack Obama, Obama administration, Michelle Obama, Affordable Care Act, or another Obama-specific reference. Do not ask generic United States government questions unless they explicitly anchor to Obama.",
    "people": "Use human, people, the human body, Homo sapiens, or another explicitly human reference. Do not ask generic mammal or animal questions.",
    "dogs": "Use dog, canine, puppy, barking, scent, or named dog breeds. Do not ask generic pet or mammal questions that could apply to cats or people.",
    "paris": "Use Paris, the Seine, Eiffel Tower, Louvre, arrondissement, or another Paris-specific reference. Do not ask generic France or Europe questions.",
    "lasers": "Use laser, laser beam, stimulated emission, coherent light, LIDAR, or named laser types. Do not ask generic optics or light questions.",
    "united_states": "Use United States, U.S., US, American, Constitution, Congress, or another explicitly United States reference. Do not ask generic politics or geography questions that could fit Obama, Paris, or another country.",
    "the_moon": "Use the Moon, lunar, Apollo, crater, regolith, or another Moon-specific reference. Do not ask generic astronomy questions that could apply to planets, space, or Earth.",
    "chess": "Use chess, checkmate, bishop, castling, FIDE, Elo, or another chess-specific reference. Do not ask generic board game or strategy questions.",
}

CONCEPT_ANCHORS = {
    "bacteria": ["bacteria", "bacterial", "bacterium", "gram-", "plasmid", "biofilm"],
    "cats": ["cat", "cats", "feline", "kitten", "purr", "whisker"],
    "obama": ["obama", "barack", "michelle obama", "affordable care act", "obama administration"],
    "people": ["human", "humans", "people", "person", "homo sapiens"],
    "dogs": ["dog", "dogs", "canine", "puppy", "bark"],
    "paris": ["paris", "seine", "eiffel", "louvre", "arrondissement"],
    "lasers": ["laser", "lasers", "lidar", "stimulated emission", "coherent light"],
    "united_states": ["united states", "u.s.", " us ", "american", "constitution", "congress"],
    "the_moon": ["the moon", " moon ", "lunar", "apollo", "regolith", "crater"],
    "chess": ["chess", "checkmate", "bishop", "castling", "fide", "elo"],
}


def make_prompt(concept: str, subtopic: str) -> str:
    guidance = CONCEPT_GUIDANCE[concept]
    return (
        f"Generate 20 unique question-answer pairs about the concept '{concept}' "
        f"and specifically about this subtopic: {subtopic}.\n\n"
        "Requirements:\n"
        "- Every question must be answerable from general factual knowledge.\n"
        "- Every question must be uniquely identifiable to the target concept when read alone.\n"
        "- Avoid vague phrasing such as 'this president', 'this country', 'this city', or 'this animal'.\n"
        "- Use the concept name itself or another unmistakable concept-specific anchor in every question.\n"
        "- Questions must be short, direct, and factual.\n"
        "- Answers should be short factual strings, usually 1-8 words.\n"
        f"- Concept-specific guidance: {guidance}\n\n"
        "Return exactly 20 items with fields 'question' and 'answer'."
    )


def flatten_to_rows(concept: str, qs: QASet) -> list[dict]:
    return [
        {
            "concept": concept,
            "question": item.question,
            "answer": item.answer,
        }
        for item in [getattr(qs, f"q{i}") for i in range(1, 21)]
    ]


def question_has_anchor(concept: str, question: str) -> bool:
    normalized = f" {question.strip().lower()} "
    return any(anchor in normalized for anchor in CONCEPT_ANCHORS[concept])


def save_csv(all_rows: list[dict]):
    RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(all_rows).to_csv(RAW_CSV, index=False)

In [ ]:
concepts = list(TOPICS.keys())[:MAX_CONCEPTS]
all_rows = []

for concept in concepts:
    subtopics = TOPICS[concept][:MAX_SUBTOPICS]
    prompts = [make_prompt(concept, sub) for sub in subtopics]
    models = [QASet] * len(subtopics)

    results = await llm.batch_respond(
        prompts=prompts,
        response_models=models,
        system="You write compact factual QA datasets. Return exactly 20 concept-specific question-answer pairs per request.",
        desc=concept,
        max_retries=200,
    )

    for result in results:
        all_rows.extend(flatten_to_rows(concept, result))

save_csv(all_rows)

In [7]:
df = pd.read_csv(RAW_CSV)
print(df.shape)
print(df.groupby("concept").size())

(10000, 3)
concept
bacteria         1000
cats             1000
chess            1000
dogs             1000
lasers           1000
obama            1000
paris            1000
people           1000
the_moon         1000
united_states    1000
dtype: int64


## Cleaning

In [ ]:
df = pd.read_csv(RAW_CSV)
df["question"] = df["question"].fillna("").str.strip()
df["answer"] = df["answer"].fillna("").str.strip()

df = df[(df["question"] != "") & (df["answer"] != "")]
df = df[df.apply(lambda row: question_has_anchor(row["concept"], row["question"]), axis=1)]
df = df.drop_duplicates(subset=["concept", "question"]).reset_index(drop=True)

In [ ]:
df.to_csv(OUT_CSV, index=False)

In [ ]:
print(df.shape)
print(df.groupby("concept").size().sort_index())

In [ ]:
df.sample(10, random_state=42)

## Subsets

In [ ]:
# 1. Drop missing rows and keep only concept-identifiable questions
# 2. Balance by concept using the smallest concept size
# 3. Split into 80:20 train/test
# 4. Save train.csv / test.csv

In [ ]:
# 1-2: drop nans, keep identifiable questions, balance by concept
df = pd.read_csv(OUT_CSV).dropna()
df = df[df.apply(lambda row: question_has_anchor(row["concept"], row["question"]), axis=1)]
min_n = df.groupby("concept").size().min()
balanced = pd.concat(
    g.sample(min_n, random_state=42) for _, g in df.groupby("concept")
).reset_index(drop=True)

In [ ]:
# 3-4: 80:20 train/test split and save
train_df, test_df = train_test_split(balanced, test_size=0.2, random_state=42)

train_df.to_csv(STORE / "train.csv", index=False)
test_df.to_csv(STORE / "test.csv", index=False)

In [ ]:
print(min_n)

In [ ]:
train_df

In [ ]:
test_df

In [ ]:
df_names = pd.read_csv(OUT_CSV)
names = sorted(df_names["concept"].dropna().astype(str).str.strip().unique())

for name in names:
    print(name)